In [1]:
import sqlite3
import pandas as pd

# Establish SQLite Database Connection
conn = sqlite3.connect("ecommerce_warehouse.db")

file_path = "ecommerce_etl.csv"
print(f"[INFO] Ingesting raw dataset from '{file_path}'...")

# Load raw CSV and save to staging table in SQLite
df = pd.read_csv(file_path)
df.to_sql("raw_sales", conn, if_exists="replace", index=False)

print("[INFO] Data extraction completed. Dataset shape:", df.shape)
print(df.head(3))

[INFO] Ingesting raw dataset from 'ecommerce_etl.csv'...
[INFO] Data extraction completed. Dataset shape: (2000, 11)
    Order_ID  Order_Date Customer_ID Customer_Name        City   Join_Date  \
0  ORD-10000  2025-01-01   CUST-1095   Customer_96  Chittagong  2024-08-19   
1  ORD-10001  2025-01-01   CUST-1151  Customer_152       Dhaka  2025-01-03   
2  ORD-10002  2025-01-01   CUST-1150  Customer_151      Khulna  2025-01-01   

  Product_ID        Category  Unit_Price  Quantity  Discount_Percent  
0   PROD-515        Clothing      321.84         3              0.10  
1   PROD-502  Home & Kitchen       33.55         1              0.15  
2   PROD-544           Books      278.51         5              0.10  


In [2]:
print(
    "[INFO] Transforming raw data and constructing Star Schema using SQL...\n"
)

# A. Dimension Table: Customers (Extracted using SQL SELECT DISTINCT)
query_customers = """
SELECT DISTINCT Customer_ID, Customer_Name, City, Join_Date 
FROM raw_sales;
"""
dim_customers = pd.read_sql_query(query_customers, conn)
dim_customers.to_sql("dim_customers", conn, if_exists="replace", index=False)
print("[INFO] Dimension Table: Customers created via SQL.")

# B. Dimension Table: Products (Extracted using SQL SELECT DISTINCT)
query_products = """
SELECT DISTINCT Product_ID, Category, Unit_Price 
FROM raw_sales;
"""
dim_products = pd.read_sql_query(query_products, conn)
dim_products.to_sql("dim_products", conn, if_exists="replace", index=False)
print("[INFO] Dimension Table: Products created via SQL.")

# C. Fact Table: Sales Performance (Financial metrics calculated via SQL)
query_fact_sales = """
SELECT 
    Order_ID,
    Order_Date,
    Customer_ID,
    Product_ID,
    Quantity,
    Discount_Percent,
    ROUND(Quantity * Unit_Price, 2) AS Gross_Amount,
    ROUND((Quantity * Unit_Price) * (1 - Discount_Percent), 2) AS Net_Sales
FROM raw_sales;
"""
fact_sales = pd.read_sql_query(query_fact_sales, conn)
fact_sales.to_sql("fact_sales", conn, if_exists="replace", index=False)
print("[INFO] Fact Table: Sales successfully built via SQL.")

# Close database connection
conn.close()

[INFO] Transforming raw data and constructing Star Schema using SQL...

[INFO] Dimension Table: Customers created via SQL.
[INFO] Dimension Table: Products created via SQL.
[INFO] Fact Table: Sales successfully built via SQL.


In [3]:

db_name = 'ecommerce_warehouse.db'
print(f"[INFO] Connecting to target Data Warehouse ('{db_name}')...")

with sqlite3.connect(db_name) as conn:
    print("[INFO] Loading 'dim_customers' table into SQLite...")
    dim_customers.to_sql('dim_customers', conn, if_exists='replace', index=False)
    
    print("[INFO] Loading 'dim_products' table into SQLite...")
    dim_products.to_sql('dim_products', conn, if_exists='replace', index=False)
    
    print("[INFO] Loading 'fact_sales' table into SQLite...")
    fact_sales.to_sql('fact_sales', conn, if_exists='replace', index=False)

print("[INFO] Data loading completed successfully.")

[INFO] Connecting to target Data Warehouse ('ecommerce_warehouse.db')...
[INFO] Loading 'dim_customers' table into SQLite...
[INFO] Loading 'dim_products' table into SQLite...
[INFO] Loading 'fact_sales' table into SQLite...
[INFO] Data loading completed successfully.


In [4]:

print("[INFO] Executing analytics query on Data Warehouse...")

analytics_query = """
SELECT 
    c.City, 
    COUNT(s.Order_ID) AS Total_Orders, 
    ROUND(SUM(s.Net_Sales), 2) AS Total_Revenue
FROM fact_sales s
JOIN dim_customers c ON s.Customer_ID = c.Customer_ID
GROUP BY c.City
ORDER BY Total_Revenue DESC;
"""

with sqlite3.connect(db_name) as conn:
    report_df = pd.read_sql_query(analytics_query, conn)

print("[INFO] Analytics query completed successfully.\n")
print("--- Revenue Performance by City ---")
print(report_df.to_string(index=False))

[INFO] Executing analytics query on Data Warehouse...
[INFO] Analytics query completed successfully.

--- Revenue Performance by City ---
      City  Total_Orders  Total_Revenue
     Dhaka           849      670795.48
Chittagong           358      277858.03
    Sylhet           304      253292.99
    Khulna           219      179150.90
  Rajshahi           178      138730.92
   Barisal            92       74788.73
